# Entrenamiento v3 — Comida Mexicana

**Modelo base:** EfficientNet-B0, 87.53% Top-1 (101 clases Food-101)

**Objetivo:** Agregar 20 clases de comida mexicana:
caldo de pollo, tacos dorados, sopes, tamales, pozole, chilaquiles,
gorditas, tlacoyos, elote, torta mexicana, molletes, huevos rancheros,
arroz rojo, frijoles, carne en su jugo, birria, menudo, flautas, mole, chiles rellenos

**Pasos:**
1. Descargar imágenes de comida mexicana (DuckDuckGo)
2. Combinar con Food-101 (101 + 20 = 121 clases)
3. Fine-tune desde modelo v2

**Instrucciones:** T4 GPU → Ejecutar todo

In [ ]:
# 1. Verificar GPU
import torch
assert torch.cuda.is_available(), "GPU no detectada"
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Clonar repo
!git clone https://github.com/Fernando-Alvarado-Soria/app-caloriasv2.git
%cd app-caloriasv2

In [ ]:
# 3. Instalar dependencias
!pip install torch torchvision numpy duckduckgo_search Pillow --quiet

In [ ]:
# 4. Descargar imágenes de comida mexicana (~150 por clase)
# Esto tarda ~10-15 minutos
from ml.collect_images import collect_all, MEXICAN_CLASSES

print(f"Clases a recolectar: {len(MEXICAN_CLASSES)}")
for cls in MEXICAN_CLASSES:
    print(f"  - {cls}")
print()

stats = collect_all(per_class=150)

In [ ]:
# 5. Verificar imágenes descargadas
import os
mexican_dir = 'ml/data/mexican_food'
total_images = 0
for cls in sorted(os.listdir(mexican_dir)):
    cls_dir = os.path.join(mexican_dir, cls)
    if os.path.isdir(cls_dir):
        n = len([f for f in os.listdir(cls_dir) if f.endswith(('.jpg','.jpeg','.png'))])
        total_images += n
        print(f"  {cls:<25} {n} imágenes")
print(f"\nTotal: {total_images} imágenes de comida mexicana")

In [ ]:
# 6. Configurar entrenamiento
import ml.config as config

config.BATCH_SIZE = 64
config.NUM_WORKERS = 2
config.DEVICE = "cuda"
config.NUM_EPOCHS = 15

# Fine-tuning completo desde el inicio
config.FREEZE_BACKBONE_EPOCHS = 0
config.UNFREEZE_AFTER = 0

# LR bajo para fine-tuning
config.LEARNING_RATE = 3e-4
config.LR_BACKBONE = 3e-5

# Regularización
config.LABEL_SMOOTHING = 0.1
config.MIXUP_ALPHA = 0.2
config.RANDOM_ERASING_PROB = 0.25
config.EARLY_STOP_PATIENCE = 6
config.CHECKPOINT_EVERY = 5

print("Configuración lista")

In [ ]:
# 7. Entrenar con dataset combinado (Food-101 + Mexican Food)
import os
import sys
import time
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, models

from ml.config import (
    DATA_DIR, MODELS_DIR, IMAGE_SIZE, NUM_WORKERS,
    RANDOM_CROP_SCALE, COLOR_JITTER, RANDOM_ROTATION,
    RANDOM_ERASING_PROB, MIXUP_ALPHA, LABEL_SMOOTHING,
    LEARNING_RATE, LR_BACKBONE, WEIGHT_DECAY,
    EARLY_STOP_PATIENCE, CHECKPOINT_EVERY,
)
from ml.train import (
    build_transforms, get_device, mixup_data, mixup_criterion,
    evaluate, save_checkpoint,
)
from ml.custom_dataset import CombinedFoodDataset

device = get_device()
print(f"Device: {device}")

# ─── Transforms ───
train_transform, val_transform = build_transforms()

# ─── Dataset combinado ───
mexican_dir = 'ml/data/mexican_food'

print("\nCargando dataset combinado...")
train_dataset = CombinedFoodDataset(
    root=DATA_DIR, split='train', transform=train_transform,
    download=True, custom_dir=mexican_dir,
)
val_dataset = CombinedFoodDataset(
    root=DATA_DIR, split='test', transform=val_transform,
    download=True, custom_dir=mexican_dir,
)

NUM_CLASSES = train_dataset.num_classes
print(f"\nTotal clases: {NUM_CLASSES}")
print(f"  Food-101: {len(train_dataset.base_classes)}")
print(f"  Mexicanas: {len(train_dataset.custom_classes)}")
print(f"  Nuevas: {train_dataset.custom_classes}")

train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE,
                          shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE,
                        shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# ─── Modelo: cargar v2 y expandir clasificador ───
print("\nCargando modelo v2 (87.53%)...")
# Construir modelo con 101 clases (original)
from ml.train import build_model
model = build_model(device)  # 101 clases

# Cargar pesos del modelo v2
scripted = torch.jit.load('ml/models/export/model_scripted.pt', map_location=device)
model.load_state_dict(scripted.state_dict())
print("Pesos v2 cargados")

# Expandir clasificador de 101 → NUM_CLASSES
if NUM_CLASSES > 101:
    old_classifier = model.classifier
    in_features = old_classifier[1].in_features
    
    # Nuevo clasificador con más clases
    new_classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, NUM_CLASSES),
    ).to(device)
    
    # Copiar pesos de las 101 clases originales
    with torch.no_grad():
        new_classifier[1].weight[:101] = old_classifier[1].weight
        new_classifier[1].bias[:101] = old_classifier[1].bias
        # Las nuevas clases se inicializan random (default)
    
    model.classifier = new_classifier
    print(f"Clasificador expandido: 101 → {NUM_CLASSES} clases")

# ─── Entrenamiento ───
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
best_val_acc = 0.0
patience_counter = 0

# Guardar lista de clases
os.makedirs(MODELS_DIR, exist_ok=True)
classes_path = os.path.join(MODELS_DIR, 'classes.txt')
with open(classes_path, 'w') as f:
    for cls in train_dataset.classes:
        f.write(cls + '\n')
print(f"Clases guardadas: {classes_path} ({NUM_CLASSES} clases)")

print(f"\n{'='*60}")
print(f"ENTRENANDO: {config.NUM_EPOCHS} épocas, {NUM_CLASSES} clases")
print(f"{'='*60}\n")

for epoch in range(config.NUM_EPOCHS):
    t0 = time.time()
    model.train()
    
    # Optimizer con LR diferenciado
    classifier_params = []
    backbone_params = []
    for name, param in model.named_parameters():
        if 'classifier' in name:
            classifier_params.append(param)
        else:
            backbone_params.append(param)
    
    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': LR_BACKBONE},
        {'params': classifier_params, 'lr': LEARNING_RATE},
    ], weight_decay=WEIGHT_DECAY)
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=len(train_loader), eta_min=1e-6
    )
    
    # Train loop
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        # Mixup
        if MIXUP_ALPHA > 0:
            images, targets_a, targets_b, lam = mixup_data(images, labels, MIXUP_ALPHA)
            optimizer.zero_grad()
            outputs = model(images)
            loss = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)
        else:
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        if (batch_idx + 1) % 100 == 0:
            print(f"  Epoch {epoch+1} | Batch {batch_idx+1}/{len(train_loader)} | "
                  f"Loss: {loss.item():.4f} | Acc: {100.*correct/total:.1f}%")
    
    train_loss = running_loss / total
    train_acc = 100. * correct / total
    
    # Validate
    val_loss, val_acc, val_top5 = evaluate(model, val_loader, criterion, device)
    
    elapsed = time.time() - t0
    print(f"\nÉpoca {epoch+1}/{config.NUM_EPOCHS} ({elapsed:.0f}s)")
    print(f"  Train — Loss: {train_loss:.4f} | Acc: {train_acc:.1f}%")
    print(f"  Val   — Loss: {val_loss:.4f} | Top-1: {val_acc:.1f}% | Top-5: {val_top5:.1f}%")
    
    # Checkpoint
    if (epoch + 1) % CHECKPOINT_EVERY == 0:
        cp_path = os.path.join(MODELS_DIR, f'checkpoint_epoch{epoch+1}.pt')
        save_checkpoint(model, optimizer, epoch, val_acc, cp_path)
    
    # Mejor modelo
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        best_path = os.path.join(MODELS_DIR, 'best_model.pt')
        save_checkpoint(model, optimizer, epoch, val_acc, best_path)
        print(f"  ★ Nuevo mejor modelo: {val_acc:.1f}%")
    else:
        patience_counter += 1
    
    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f"\nEarly stopping: sin mejora en {EARLY_STOP_PATIENCE} épocas.")
        break
    print()

print(f"\nEntrenamiento completado. Mejor Val Top-1: {best_val_acc:.1f}%")

In [ ]:
# 8. Exportar modelo con nuevas clases
import json

# Cargar mejor modelo
checkpoint = torch.load(os.path.join(MODELS_DIR, 'best_model.pt'), map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

val_acc = checkpoint.get('val_acc', 0)
epoch = checkpoint.get('epoch', '?')
print(f"Mejor modelo: época {epoch}, val_acc={val_acc:.1f}%")

# Exportar TorchScript
export_dir = os.path.join(MODELS_DIR, 'export')
os.makedirs(export_dir, exist_ok=True)

example_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(device)
scripted_model = torch.jit.trace(model, example_input)
scripted_path = os.path.join(export_dir, 'model_scripted.pt')
scripted_model.save(scripted_path)
print(f"TorchScript: {scripted_path}")

# State dict
inference_path = os.path.join(export_dir, 'model_inference.pt')
torch.save(model.state_dict(), inference_path)

# Metadata
classes = [line.strip() for line in open(classes_path)]
metadata = {
    'model_name': 'efficientnet_b0',
    'num_classes': NUM_CLASSES,
    'image_size': IMAGE_SIZE,
    'val_accuracy_top1': round(val_acc, 2),
    'epoch': epoch,
    'classes': classes,
    'normalize': {
        'mean': [0.485, 0.456, 0.406],
        'std': [0.229, 0.224, 0.225],
    },
    'custom_classes': train_dataset.custom_classes,
}
meta_path = os.path.join(export_dir, 'metadata.json')
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"Metadata: {meta_path}")
print(f"\nClases totales: {NUM_CLASSES} (101 Food-101 + {len(train_dataset.custom_classes)} mexicanas)")

In [ ]:
# 9. Descargar modelo v3
from google.colab import files
import shutil

os.makedirs('download_pack', exist_ok=True)
if os.path.exists('ml/models/export'):
    shutil.copytree('ml/models/export', 'download_pack/export', dirs_exist_ok=True)
if os.path.exists('ml/models/best_model.pt'):
    shutil.copy2('ml/models/best_model.pt', 'download_pack/best_model.pt')
if os.path.exists('ml/models/classes.txt'):
    shutil.copy2('ml/models/classes.txt', 'download_pack/classes.txt')

shutil.make_archive('modelo_v3_mexicano', 'zip', 'download_pack')
files.download('modelo_v3_mexicano.zip')
print('Descarga iniciada.')